# Phase 4 — Hybrid Path-Planning + RL Coordination Training 🔥

This notebook trains and evaluates the Phase 4 **Hybrid Path-Planning + RL Coordination** MARL models on Google Colab's GPU runtime. Low-level navigation is handled deterministically via Manhattan/A-star routing, while high-level coordination is handled by PPO.

### Step-by-Step Instructions:
1. **Upload workspace**: Zip your repository directory (exclude the `models/` folder to keep it small) as `wildfire-rl.zip` and drag-and-drop it into the Colab file panel on the left.
2. **Select Runtime**: Select **Runtime** -> **Change runtime type** -> select **T4 GPU** -> **Save**.
3. **Run all cells** sequentially.

### 1. Extract Workspace Code

In [ ]:
import os
from pathlib import Path

zip_name = "wildfire-rl.zip"
if Path(zip_name).exists():
    !unzip -q {zip_name} -d wildfire-rl
    %cd wildfire-rl
    print(f"Successfully entered workspace: {os.getcwd()}")
else:
    print(f"ERROR: Could not find '{zip_name}' in the root directory. Please upload it first.")

### 2. Install Project Dependencies

In [ ]:
!grep -v "torch" requirements.txt | grep -v "numpy" > req_clean.txt
!pip install -q -r req_clean.txt
!pip install -q -e .

### 3. Verify CUDA (GPU) Activation

In [ ]:
import torch
print(f"CUDA (GPU) Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not active. Go to Runtime -> Change runtime type and select T4 GPU.")

### 4. Run Local Hybrid Smoke Test (2k steps)

In [ ]:
# Execute the training smoke test to verify V4 environments and routers
!python -m pytest tests/test_marl_v3.py -v
!python -c "from wildfire_rl.envs.hybrid_multi_agent import HybridMultiAgentWildfireEnv; print('Hybrid Multi-Agent environment imports successfully!')"

### 5. Run Parallel Subprocess Hybrid PPO MARL Training on GPU (Matched Budgets)

Trains models for:
- Regions: **Saudi Arabia** and **California**
- Team sizes: **5** and **10** agents
- Routing strategies: **NearestFire** (`nearest_fire`) and **Frontier** (`frontier`)
- Budgets: 5 agents -> 500,000 steps | 10 agents -> 1,000,000 steps

*We use `MAX_PARALLEL = 2` to keep resource utilization stable.*

In [ ]:
import subprocess
import time
from pathlib import Path

FORCE_RETRAIN = True
MAX_PARALLEL = 2

TIMESTEPS_MAP = {
    5: 500000,
    10: 1000000
}

jobs = []
for region in ["saudi", "california"]:
    for n_agents, timesteps in TIMESTEPS_MAP.items():
        for strategy in ["nearest_fire", "frontier"]:
            for seed in [0, 1, 2]:
                jobs.append((region, n_agents, seed, timesteps, strategy))

processes = []
print(f"Launching {len(jobs)} Hybrid PPO training runs in parallel (MAX_PARALLEL={MAX_PARALLEL})...")

for region, agents, seed, timesteps, strategy in jobs:
    model_file = Path("models") / f"ppo_hybrid_marl_{region}_{agents}agents_{strategy}_seed_{seed}.zip"
    if model_file.exists() and not FORCE_RETRAIN:
        print(f"Skip: {model_file.name} already exists.")
        continue

    while len(processes) >= MAX_PARALLEL:
        for p in list(processes):
            if p.poll() is not None:
                processes.remove(p)
        time.sleep(1)

    cmd = [
        "python", "scripts/train_single_hybrid_marl.py",
        "--region", region,
        "--agents", str(agents),
        "--seed", str(seed),
        "--timesteps", str(timesteps),
        "--strategy", strategy
    ]
    print(f"Launching: {region}, {agents} agents, {strategy}, seed={seed}")
    p = subprocess.Popen(cmd)
    processes.append(p)

for p in processes:
    p.wait()

print("\nAll Hybrid PPO training runs completed successfully!")

### 6. Run Hybrid Evaluation Pipeline

In [ ]:
!python scripts/run_hybrid_evaluation.py
print("\n✅ Hybrid evaluation complete. Check results/v4/ and figures/v4/ for outputs.")

### 7. Package Checkpoints and Results

In [ ]:
!zip -j hybrid_models.zip models/ppo_hybrid_marl_*
!zip -r hybrid_results.zip results/v4/ figures/v4/

print("\n--- READY FOR DOWNLOAD ---")
print("1. Click the file explorer icon in Colab (left panel).")
print("2. Download 'hybrid_models.zip' and 'hybrid_results.zip'.")
print("3. Extract hybrid_models.zip locally into your 'models/' folder.")
print("4. Extract hybrid_results.zip into your repo root.")

### 8. expected Runtime Estimation

Based on T4 GPU benchmarks with `MAX_PARALLEL = 2`:
- **100k timesteps**: ~4 minutes per job (Total ~48 minutes if run sequentially)
- **300k timesteps**: ~12 minutes per job
- **500k timesteps** (5-agent): ~18 minutes per job
- **1M timesteps** (10-agent): ~35 minutes per job
- **Total Parallel runtime** (with 2 concurrent workers): **~3 to 4 hours**.